# Slide 21 pilot, part 2: Wan 2.2 TI2V-5B alone

Same 5 big-ball items; Wan needs the whole T4, so it runs in its own session.


In [ ]:
!pip install -q -U "diffusers>=0.35" "transformers>=4.55" accelerate sentencepiece ftfy imageio imageio-ffmpeg matplotlib

In [ ]:
import base64, io, tarfile, os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
BLOB = "..."
os.makedirs("/kaggle/working/src", exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(BLOB)), mode="r:gz").extractall("/kaggle/working/src")
del BLOB
print(os.listdir("/kaggle/working/src"))

In [ ]:
# ---- settings
RUN_VJEPA, RUN_LTX, RUN_WAN = False, False, True   # Wan alone: it needs the whole GPU
LTX_SAMPLES, WAN_SAMPLES = 2, 1
MODES = ["still", "clip"]
LTX_REPO = "Lightricks/LTX-Video-0.9.5"
WAN_REPO = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
VJEPA_REPO = "facebook/vjepa2-vitl-fpc64-256"
W, H = 768, 512
NUM_FRAMES = 49                  # LTX: 8k+1; Wan: 4k+1
LTX_STEPS, WAN_STEPS = 50, 40
CLIP_COND = 17                   # LTX clip mode: last 17 input frames (8k+1)
WAN_MAX_S = 25 * 60              # give up on Wan if one clip takes longer than this
PROMPT = ("A glossy red ball rolls quickly through a curved grey tube lying on a flat light grey table, "
          "seen from directly above. The ball rolls out of the open end of the tube and keeps rolling "
          "across the table. Smooth continuous motion, static camera, the ball keeps its size and colour.")
NEGATIVE = ("static, frozen, still image, ball disappears, ball shrinks, ball fades, morphing, tube changes shape, "
            "camera motion, blurry, distorted, text, watermark, worst quality")

import glob, json, sys, time, csv, math
import numpy as np, torch, imageio.v2 as imageio
from PIL import Image
ROOT = "/kaggle/working/src"; sys.path.insert(0, ROOT); sys.path.insert(0, ROOT + "/pilot")
import analyze
ITEMS = analyze.load_items(ROOT + "/inputs_big")
OUT = "/kaggle/working/out"
for d in ("ltx", "wan"): os.makedirs(f"{OUT}/{d}", exist_ok=True)
print(len(ITEMS), "items;", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

def frames_of(m, name):
    return np.stack(imageio.mimread(os.path.join(m["dir"], m["files"][name]), memtest=False))[..., :3]

def check(v, name):
    v = np.nan_to_num(v)
    ok = v.std() > 2
    print(f"{name}: std {v.std():.1f}", "" if ok else "BLANK")
    return ok

## 1 · V-JEPA 2

In [ ]:
if RUN_VJEPA:
    from transformers import AutoModel, AutoVideoProcessor
    vj = AutoModel.from_pretrained(VJEPA_REPO, dtype=torch.float16).cuda().eval()
    proc = AutoVideoProcessor.from_pretrained(VJEPA_REPO)
    def square(fr):
        T, h, w, _ = fr.shape
        out = np.full((T, w, w, 3), 242, np.uint8); y0 = (w - h) // 2
        out[:, y0:y0 + h] = fr
        return out
    def to_tensor(fr):
        return proc(videos=square(fr), return_tensors="pt")["pixel_values_videos"].to("cuda", torch.float16)
    METHOD = None
    @torch.no_grad()
    def surprise(fr16):
        global METHOD
        x = to_tensor(fr16)
        full = vj.get_vision_features(x); N = full.shape[1]; Nc = N // 2
        tm = [torch.arange(Nc, N, device="cuda")[None]]; cm = [torch.arange(Nc, device="cuda")[None]]
        try:
            ctx = vj.get_vision_features(x[:, :8])
            pred = vj.predictor(encoder_hidden_states=ctx, context_mask=cm, target_mask=tm).last_hidden_state
            assert pred.shape == full[:, Nc:].shape
            METHOD = METHOD or "context-only encoding + predictor"
        except Exception as e:
            fake = np.concatenate([fr16[:8], np.repeat(fr16[7:8], 8, 0)])
            pred = vj(pixel_values_videos=to_tensor(fake), context_mask=cm, target_mask=tm).predictor_output.last_hidden_state
            METHOD = METHOD or f"frozen-future fallback ({type(e).__name__})"
        return (pred.float() - full[:, Nc:].float()).abs().mean().item()
    rows = []
    for iid, m in sorted(ITEMS.items()):
        cor, imp = frames_of(m, "full"), frames_of(m, "anomalous_impetus")
        c = m["context_end_frame"]
        for mode in MODES:
            def clip16(fr):
                fut = fr[c + 1:c + 9]
                ctx = fr[c - 7:c + 1] if mode == "clip" else np.repeat(fr[c:c + 1], 8, 0)
                return np.concatenate([ctx, fut])
            rows.append(dict(item_id=iid, shape=m["params"]["shape"], speed=m["params"]["speed"], mode=mode,
                             surprise_correct=surprise(clip16(cor)), surprise_impetus=surprise(clip16(imp))))
    print("method:", METHOD)
    with open(f"{OUT}/vjepa.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    for r in rows: print(r["item_id"], r["mode"], "prefers curving" if r["surprise_impetus"] < r["surprise_correct"] else "prefers straight")
    del vj; torch.cuda.empty_cache()

## 2 · LTX-Video, 50 steps, motion prompt

In [ ]:
if RUN_LTX:
    from diffusers import LTXConditionPipeline
    from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
    def load_ltx(dtype):
        p = LTXConditionPipeline.from_pretrained(LTX_REPO, torch_dtype=dtype)
        p.text_encoder.to("cuda")
        with torch.no_grad():
            emb = p.encode_prompt(prompt=PROMPT, negative_prompt=NEGATIVE, do_classifier_free_guidance=True,
                                  device="cuda", max_sequence_length=128)
        p.text_encoder.to("cpu"); torch.cuda.empty_cache()
        p.transformer.to("cuda"); p.vae.to("cuda"); p.vae.enable_tiling()
        return p, emb
    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    pipe, EMB = load_ltx(DTYPE)
    def gen_ltx(m, mode, seed):
        if mode == "still":
            cond = LTXVideoCondition(image=Image.open(os.path.join(m["dir"], m["files"]["still"])).convert("RGB"), frame_index=0)
        else:
            ctx = frames_of(m, "context")[-CLIP_COND:]
            cond = LTXVideoCondition(video=[Image.fromarray(f) for f in ctx], frame_index=0)
        pe, pm, npe, npm = EMB
        kw = dict(conditions=[cond], prompt_embeds=pe, prompt_attention_mask=pm, negative_prompt_embeds=npe,
                  negative_prompt_attention_mask=npm, width=W, height=H, num_frames=NUM_FRAMES,
                  num_inference_steps=LTX_STEPS, guidance_scale=3.0,
                  generator=torch.Generator("cpu").manual_seed(seed), output_type="np")
        try:
            v = pipe(frame_rate=16, **kw).frames[0]
        except TypeError:
            v = pipe(**kw).frames[0]
        return (np.clip(np.nan_to_num(v), 0, 1) * 255).astype(np.uint8)
    todo = [(m, mode, j) for m in sorted(ITEMS.values(), key=lambda m: m["item_id"]) for j in range(LTX_SAMPLES) for mode in MODES]
    for i, (m, mode, j) in enumerate(todo):
        path = f"{OUT}/ltx/{m['item_id']}__{mode}__s{j}.mp4"
        if os.path.exists(path): continue
        t0 = time.time()
        v = gen_ltx(m, mode, 1000 * j + 17 * i)
        if i == 0 and not check(v, "first LTX clip"):
            del pipe; torch.cuda.empty_cache(); DTYPE = torch.float32
            pipe, EMB = load_ltx(DTYPE); v = gen_ltx(m, mode, 0)
        imageio.mimwrite(path, v, fps=16, macro_block_size=16)
        print(f"LTX {i+1}/{len(todo)} {os.path.basename(path)} {time.time()-t0:.0f}s", flush=True)
    del pipe; torch.cuda.empty_cache()

## 3 · Wan 2.2 TI2V-5B from the still

In [ ]:
if RUN_WAN:
    import gc; gc.collect()
    try:
        from diffusers import WanImageToVideoPipeline, AutoencoderKLWan
        vae = AutoencoderKLWan.from_pretrained(WAN_REPO, subfolder="vae", torch_dtype=torch.float32)
        wdt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        wan = WanImageToVideoPipeline.from_pretrained(WAN_REPO, vae=vae, torch_dtype=wdt)
        # The T5 text encoder alone is ~11 GB: encode the prompt once, then drop it, so the
        # transformer and the VAE fit on the T4 (cpu offload kept the encoder on the GPU and ran out of memory)
        wan.text_encoder.to("cuda")
        with torch.no_grad():
            PE, NPE = wan.encode_prompt(prompt=PROMPT, negative_prompt=NEGATIVE, do_classifier_free_guidance=True,
                                        device="cuda")
        wan.text_encoder.to("cpu"); wan.text_encoder = None
        gc.collect(); torch.cuda.empty_cache()
        wan.enable_model_cpu_offload()
        try: wan.vae.enable_tiling()
        except Exception: pass
        print("GPU memory after setup:", torch.cuda.memory_allocated() // 2**20, "MB")
        for i, m in enumerate(sorted(ITEMS.values(), key=lambda m: m["item_id"])):
            for j in range(WAN_SAMPLES):
                path = f"{OUT}/wan/{m['item_id']}__still__s{j}.mp4"
                if os.path.exists(path): continue
                t0 = time.time()
                img = Image.open(os.path.join(m["dir"], m["files"]["still"])).convert("RGB")
                v = wan(image=img, prompt_embeds=PE, negative_prompt_embeds=NPE, height=H, width=W, num_frames=NUM_FRAMES,
                        num_inference_steps=WAN_STEPS, guidance_scale=5.0,
                        generator=torch.Generator("cpu").manual_seed(1000 * j + i), output_type="np").frames[0]
                v = (np.clip(np.nan_to_num(v), 0, 1) * 255).astype(np.uint8)
                check(v, "Wan clip")
                imageio.mimwrite(path, v, fps=16, macro_block_size=16)
                dt = time.time() - t0
                print(f"Wan {os.path.basename(path)} {dt:.0f}s", flush=True)
                if dt > WAN_MAX_S:
                    raise RuntimeError(f"too slow on this GPU ({dt:.0f} s per clip), stopping Wan")
    except Exception as e:
        import traceback; traceback.print_exc()
        print("WAN STOPPED:", type(e).__name__, e)

## 4 · Score

In [ ]:
import subprocess
res = "/kaggle/working/results"
r = subprocess.run([sys.executable, ROOT + "/pilot/analyze_v2.py", "--inputs", ROOT + "/inputs_big",
                    "--gen", f"wan={OUT}/wan", "--cond", f"ltx_clip={CLIP_COND}",
                    "--out", res], capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
for f in [f"{OUT}/vjepa.csv", f"{res}/summary.csv", f"{res}/per_clip.csv"]:
    print("=====", f); print(open(f).read() if os.path.exists(f) else "(missing)")
!cd /kaggle/working && zip -qr slide21_pilot_results.zip results out src/inputs_big && ls -la slide21_pilot_results.zip